In [1]:
# Célula 2: Importação das bibliotecas e configuração do pipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import importlib
import metrics
importlib.reload(metrics)

C:\Users\gutol\AppData\Local\Temp\ipykernel_7396\2801408908.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


<module 'metrics' from 'c:\\Users\\gutol\\Downloads\\Ollama\\Nova pasta\\metrics.py'>

In [2]:
# ==========================================
# 1. CARREGAMENTO E DIVISÃO DO PDF
# ==========================================
# Substitua pelo caminho real do seu arquivo PDF no computador
caminho_pdf = "C:\\Users\\gutol\\OneDrive\\Documentos\\Mestrado\\LLM\\Regulamento_Estagio_UFRB.pdf"

print(f"Carregando o arquivo: {caminho_pdf}...")
loader = PyPDFLoader(caminho_pdf)
documentos_carregados = loader.load()


Carregando o arquivo: C:\Users\gutol\OneDrive\Documentos\Mestrado\LLM\Regulamento_Estagio_UFRB.pdf...


In [3]:

# Como PDFs podem ser gigantescos, dividimos o texto em blocos (chunks) menores
# chunk_size: tamanho do bloco | chunk_overlap: sobreposição para não cortar contextos no meio
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documentos_fragmentados = text_splitter.split_documents(documentos_carregados)
print(f"PDF dividido em {len(documentos_fragmentados)} blocos de texto.")



PDF dividido em 71 blocos de texto.


In [4]:

# ==========================================
# 2. EMBEDDINGS E BANCO DE VETORES
# ==========================================
diretorio_db = "./data_documents"
# Criando IDs únicos baseados no nome do arquivo e número do pedaço


embeddings = OllamaEmbeddings(model="nomic-embed-text")
ids = [f"doc_{caminho_pdf.split('.')[0]}_chunk_{i}" for i in range(len(documentos_fragmentados))]

print("Indexando blocos do PDF no banco de vetores local...")
# Criamos o banco a partir dos documentos fragmentados extraídos do PDF
vectorstore = Chroma.from_documents(
    documents=documentos_fragmentados, 
    embedding=embeddings,
    persist_directory=diretorio_db,
    ids=ids
)

#preparação para a pergunta com similaridade de 3 trechos mais relevantes parecidos
# Configura o buscador para trazer os 3 blocos mais relevantes para a pergunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Indexando blocos do PDF no banco de vetores local...


In [5]:

# ==========================================
# 3. CONFIGURAÇÃO DO LLM E PROMPT RAG
# ==========================================

# temperature indica o nível de criatividade da LLM, ou grau de liberdade


template_rag = """
[CONTEXTO]
Utilize estritamente as informações extraídas do regulamento de estágio da UFRB em PDF fornecidas abaixo para responder à pergunta do usuário de forma clara e direta. 
Se a resposta não puder ser encontrada nos fragmentos, diga apenas que a informação não consta no documento.

FRAGMENTOS DO PDF:
{context}

[PERGUNTA]
{question}

[RESPOSTA]
"""
prompt = ChatPromptTemplate.from_template(template_rag)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construção da cadeia RAG
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt

    | StrOutputParser()
)

print("Sistema de RAG com suporte a PDF pronto!")

Sistema de RAG com suporte a PDF pronto!


In [6]:
perguntas = ["Quais documentos são necessários para redução de carga horária?", 
             "Existe alguma hipótese em que o estudante possa ser dispensado do estágio supervisionado obrigatório?"]


In [7]:
gabaritos = [
"""I-Comprovante de vínculo empregatício (cópia da Carteira de Trabalho ou cópia de nomeação no Diário Oficial);
II - Três últimos contracheques;
III - Atestado de frequência da escola;
IV - Relatório da Coordenação de Área ou Coordenação Pedagógica.""",

"""Sob nenhuma hipótese o estudante será dispensado do Estágio Supervisionado
Obrigatório, nem mesmo será permitida a realização de atividades domiciliares por
motivo de doença ou licença maternidade. Nestes casos, o estudante não poderá se
matricular. Entretanto, caso ele esteja cursando o componente curricular Estágio
Supervisionado deverá solicitar o trancamento do mesmo e se matricular em outro
semestre, no prazo estipulado pela Universidade. """
]

In [8]:
import time
import pandas as pd

from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ======================================================
# MODELOS A SEREM TESTADOS
# ======================================================

modelos = [
    "gemma:7b",
    "llama3.1:8b",
    "deepseek-r1:7b"
]

# ======================================================
# FUNÇÃO PARA CARREGAR A LLM
# ======================================================

def carregar_llm(modelo):

    return ChatOllama(
        model=modelo,
        temperature=0
    )


# ======================================================
# FUNÇÃO PARA CRIAR A RAG CHAIN
# ======================================================

def criar_rag_chain(llm):

    return (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )


# ======================================================
# EXECUTAR TODOS OS MODELOS
# ======================================================

resultados = []

resultados = []

for modelo in modelos:

    print("\n" + "="*100)
    print(f"MODELO: {modelo}")
    print("="*100)

    llm = carregar_llm(modelo)

    rag_chain = criar_rag_chain(llm)

    for i, pergunta in enumerate(perguntas):

        print(f"Modelo: {modelo} | Pergunta {i+1}")

        print(f"\nPergunta: {pergunta}")

        inicio = time.time()

        # Gera a resposta da LLM
        resposta = rag_chain.invoke(pergunta)

        fim = time.time()

        tempo = round(fim - inicio, 2)


        # Calcula as métricas operacionais
        metricas = metrics.calcular_metricas_operacionais(
            prompt=pergunta,
            resposta=resposta,
            tempo_inicio=inicio,
            tempo_fim=fim
        )


        # Calcula o ROUGE
        rouge = metrics.calcular_rouge(
            resposta,
            gabaritos[i]
        )


        # Avaliação da LLM Juiz
        avaliacao = metrics.llmAsJudge(
            pergunta,
            resposta,
            gabaritos[i],
            "llama3.1:8b"
        )
        print("\n===== RETORNO DO JUIZ =====")
        print(avaliacao)
        print(type(avaliacao))
        print(avaliacao.keys())
        print("===========================\n")


        resultados.append({

            "Modelo": modelo,
            "Pergunta": pergunta,
            "Tempo (s)": tempo,
            "Tokens": metricas["tokens_totais"],
            "Tokens/s": metricas["tokens_por_segundo"],
            "ROUGE-1": rouge["rouge1_f1"],
            "ROUGE-2": rouge["rouge2_f1"],
            "ROUGE-L": rouge["rougeL_f1"],
            "Nota Juiz": avaliacao["nota_final"],
            "Comentário": avaliacao["comentario"],
            "Resposta": resposta

        })


        print("\nResposta:")
        print(resposta)
        print("-"*100)
# ======================================================
# GERAR TABELA
# ======================================================

df = pd.DataFrame(resultados)

display(df)
# ======================================================
# GERAR TABELA
# ======================================================

df = pd.DataFrame(resultados)

display(df)

# ======================================================
# RANKING GERAL POR MODELO
# ======================================================

ranking = (
    df.groupby("Modelo")
      .agg({
          "Nota Juiz": "mean",
          "ROUGE-1": "mean",
          "ROUGE-2": "mean",
          "ROUGE-L": "mean",
          "Tempo (s)": "mean",
          "Tokens/s": "mean"
      })
      .round(3)
      .reset_index()
)

ranking = ranking.sort_values(
    by=["Nota Juiz", "ROUGE-L", "Tempo (s)"],
    ascending=[False, False, True]
).reset_index(drop=True)

ranking.insert(0, "Ranking", range(1, len(ranking)+1))

display(ranking)
# ======================================================
# SALVAR RESULTADOS
# ======================================================
with pd.ExcelWriter("Resultado_LLMs.xlsx") as writer:

    df.to_excel(writer,
                sheet_name="Resultados",
                index=False)

    ranking.to_excel(writer,
                     sheet_name="Ranking Geral",
                     index=False)

#df.to_excel("Resultado_LLMs.xlsx", index=False)

#print("\nArquivo Resultado_LLMs.xlsx criado com sucesso.")


MODELO: gemma:7b
Modelo: gemma:7b | Pergunta 1

Pergunta: Quais documentos são necessários para redução de carga horária?
========== RESPOSTA DO JUIZ ==========
{
"correção":8,
"completude":6,
"clareza":9,
"nota_final":7.67,
"comentário":"A resposta da IA apresenta boa correção, mas faltou incluir o item III do gabarito (Atestado de frequência da escola). Além disso, a inclusão do Relatório da Coordenação de Área ou Coordenação Pedagógica foi feita com uma variante que não está presente no gabarito. A clareza da resposta é boa, mas poderia ser melhorada com mais detalhes sobre o processo de redução de carga horária."
}

===== RETORNO DO JUIZ =====
{'correção': 8, 'completude': 6, 'clareza': 9, 'nota_final': 7.67, 'comentário': 'A resposta da IA apresenta boa correção, mas faltou incluir o item III do gabarito (Atestado de frequência da escola). Além disso, a inclusão do Relatório da Coordenação de Área ou Coordenação Pedagógica foi feita com uma variante que não está presente no gabar

,Modelo,Pergunta,Tempo (s),Tokens,Tokens/s,ROUGE-1,ROUGE-2,ROUGE-L,Nota Juiz,Comentário,Resposta
0,gemma:7b,Quais documentos são necessários para redução ...,712.85,139,0.18,0.6515,0.5846,0.6515,7.67,,Os seguintes documentos são necessários para r...
1,gemma:7b,Existe alguma hipótese em que o estudante poss...,373.32,111,0.23,0.7097,0.6393,0.6613,7.70,,A resposta está no parágrafo 3° do fragmento d...
2,llama3.1:8b,Quais documentos são necessários para redução ...,405.25,196,0.45,0.5647,0.5357,0.5647,8.30,,De acordo com o regulamento de estágio da UFRB...
3,llama3.1:8b,Existe alguma hipótese em que o estudante poss...,245.89,87,0.25,0.5225,0.3486,0.4144,7.70,,Não consta no documento. Apenas é mencionada a...
4,deepseek-r1:7b,Quais documentos são necessários para redução ...,310.70,156,0.46,0.6294,0.4965,0.5734,8.70,,\n\nOs documentos necessários para redução de ...
5,deepseek-r1:7b,Existe alguma hipótese em que o estudante poss...,177.88,59,0.19,0.3871,0.2637,0.2581,7.67,"A resposta da IA está próxima do gabarito, mas...",\n\nNão há hipótese mencionada no regulamento ...


,Modelo,Pergunta,Tempo (s),Tokens,Tokens/s,ROUGE-1,ROUGE-2,ROUGE-L,Nota Juiz,Comentário,Resposta
0,gemma:7b,Quais documentos são necessários para redução ...,712.85,139,0.18,0.6515,0.5846,0.6515,7.67,,Os seguintes documentos são necessários para r...
1,gemma:7b,Existe alguma hipótese em que o estudante poss...,373.32,111,0.23,0.7097,0.6393,0.6613,7.70,,A resposta está no parágrafo 3° do fragmento d...
2,llama3.1:8b,Quais documentos são necessários para redução ...,405.25,196,0.45,0.5647,0.5357,0.5647,8.30,,De acordo com o regulamento de estágio da UFRB...
3,llama3.1:8b,Existe alguma hipótese em que o estudante poss...,245.89,87,0.25,0.5225,0.3486,0.4144,7.70,,Não consta no documento. Apenas é mencionada a...
4,deepseek-r1:7b,Quais documentos são necessários para redução ...,310.70,156,0.46,0.6294,0.4965,0.5734,8.70,,\n\nOs documentos necessários para redução de ...
5,deepseek-r1:7b,Existe alguma hipótese em que o estudante poss...,177.88,59,0.19,0.3871,0.2637,0.2581,7.67,"A resposta da IA está próxima do gabarito, mas...",\n\nNão há hipótese mencionada no regulamento ...


,Ranking,Modelo,Nota Juiz,ROUGE-1,ROUGE-2,ROUGE-L,Tempo (s),Tokens/s
0,1,deepseek-r1:7b,8.185,0.508,0.380,0.416,244.290,0.325
1,2,llama3.1:8b,8.000,0.544,0.442,0.490,325.570,0.350
2,3,gemma:7b,7.685,0.681,0.612,0.656,543.085,0.205


In [9]:
print("\n--- Avaliação do Juiz ---")
print(f"Pergunta: {pergunta}")
print(f"Modelo avaliado: {modelo}")
print(avaliacao)


--- Avaliação do Juiz ---
Pergunta: Existe alguma hipótese em que o estudante possa ser dispensado do estágio supervisionado obrigatório?
Modelo avaliado: deepseek-r1:7b
{'correcao': 8, 'completude': 6, 'clareza': 9, 'nota_final': 7.67, 'comentario': 'A resposta da IA está próxima do gabarito, mas não menciona a possibilidade de trancamento do componente curricular Estágio Supervisionado em caso de doença ou licença maternidade.'}


In [11]:
# ======================================================
# EXPORTAR PARA EXCEL
# ======================================================

arquivo = r"C:\Users\gutol\Downloads\Ollama\Resultado_LLMs.xlsx"

with pd.ExcelWriter(arquivo) as writer:

    # Aba 1 - Resultados detalhados
    df.to_excel(
        writer,
        sheet_name="Resultados",
        index=False
    )

    # Aba 2 - Ranking geral
    ranking.to_excel(
        writer,
        sheet_name="Ranking Geral",
        index=False
    )

print(f"\nArquivo salvo com sucesso em:\n{arquivo}")


Arquivo salvo com sucesso em:
C:\Users\gutol\Downloads\Ollama\Resultado_LLMs.xlsx
